# Value at Risk y Expected Shortfall · el 1 % de días que te liquida
**Explicación de un matemático** · Julián Castillo, trader cuantitativo de Castle Capital

Este Colab tiene dos partes:

- **Parte 1 · Las gráficas del video.** Datos reales de BTC/USD desde 2020: cuánto cae en un día malo, cuánto cae cuando ese día malo se queda corto, y qué le pasa a una cuenta apalancada.
- **Parte 2 · Tu calculadora.** Pones el activo que operas, el tamaño de tu cuenta y cuánto aceptas perder en un día malo, y te devuelve tu VaR, tu Expected Shortfall y tu **posición máxima**.

**Cómo usarlo:** menú *Entorno de ejecución → Ejecutar todo*. Después cambia los valores de la Parte 2 y vuelve a ejecutar esa celda. Necesita conexión a internet (descarga precios y tipografías).

| Figura | Momento del video | Frase en la que entra |
|---|---|---|
| 1 | Matemática (0:21) | "el noventa y nueve por ciento de los días cae menos de nueve" |
| 2 | Matemática y nombre (0:28–0:41) | "cae en promedio más de doce" · "Ese nueve es el Value at Risk…" |
| 3 | Matemática (0:32) | "Y en el crash del covid, treinta y siete. Liquidado." |
| 4 | Veredicto (0:41) | "no es mala suerte: eso pasa unas tres veces al año" |
| 5 | Control (0:46) | "Si aceptas perder dos por ciento, el crash del covid te cuesta seis. No la cuenta." |
| 6 y 7 | Calculadora | — |

**Color:** la primera celda de código tiene el selector `OPCION`: `'imperial'` (tonos profundos sobre Oxford White, por defecto) o `'signal'` (tonos claros sobre blanco).

*Contenido educativo, no es recomendación de inversión.*

In [ ]:
import os, urllib.request, logging
logging.getLogger('matplotlib.font_manager').setLevel(logging.ERROR)   # silencia avisos de fuentes de respaldo
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.patches import FancyBboxPatch
from matplotlib import font_manager as fm
from matplotlib.colors import LinearSegmentedColormap

# =====================  ELIGE LA OPCIÓN DE COLOR  =====================
OPCION = 'imperial'   # 'imperial' = tonos profundos sobre Oxford White · 'signal' = tonos claros sobre blanco
# ======================================================================

# --- Paletas Castle Capital (Design System, oct. 2026). Siempre fondo claro. ---
PALETAS = {
    'imperial': dict(
        BG='#F4F4F0',        # Oxford White
        PANEL_BG='#FFFFFF',  # White (cajas, leyendas)
        FG='#3D3D3D',        # Oxford Dark (texto)
        TITLE='#1B3A6B',     # Imperial Blue (titulares)
        GRID='#E2E2DB', SPINE='#C9C9C0',
        MAIN='#1B3A6B',      # Imperial Blue: serie protagonista / lado correcto
        ALERT='#C97A3A',     # Orange Dark: contraste / lado riesgoso
        SECOND='#2B5497',    # Castle Blue: serie secundaria
        GOLD='#C9A84C',      # Castle Gold: el único acento (umbral, valor clave)
        GOLD_TXT='#8C7325',  # Gold Deep: oro legible para texto sobre claro
        MUTED='#6B6B6B',     # Oxford Grey: texto secundario
    ),
    'signal': dict(
        BG='#FFFFFF',        # White
        PANEL_BG='#E8EDF5',  # Arctic
        FG='#3D3D3D',
        TITLE='#1B3A6B',
        GRID='#E8EDF5', SPINE='#D5DCE8',
        MAIN='#4A8FD4',      # Signal Blue
        ALERT='#E8954F',     # Castle Orange
        SECOND='#2B5497',    # Castle Blue
        GOLD='#C9A84C',
        GOLD_TXT='#8C7325',
        MUTED='#6B6B6B',
    ),
}
P = PALETAS[OPCION]
BG, PANEL_BG, FG, TITLE, GRID = P['BG'], P['PANEL_BG'], P['FG'], P['TITLE'], P['GRID']
MAIN, ALERT, SECOND, GOLD, GOLD_TXT, MUTED = P['MAIN'], P['ALERT'], P['SECOND'], P['GOLD'], P['GOLD_TXT'], P['MUTED']
POSITIVE, NEGATIVE = '#4CAF50', '#EF5350'   # solo cuando la figura es literalmente ganancia vs. pérdida
GREEN, PINK, NEUTRAL = MAIN, ALERT, SECOND   # alias para las plantillas de chart_library.md
DIV_CMAP = LinearSegmentedColormap.from_list('castle_div', [ALERT, '#F4F4F0', MAIN])
SEQ_CMAP = LinearSegmentedColormap.from_list('castle_seq', ['#E8EDF5', '#1B3A6B'])
CRYPTO_CMAP = DIV_CMAP                       # alias para plantillas antiguas

# --- Tipografías de marca: Cormorant Garamond (titulares), DM Sans (texto), Barlow (cifras) ---
FONT_DIR = '/content/castle_fonts' if os.path.isdir('/content') else os.path.expanduser('~/.castle_fonts')
CDN = 'https://cdn.jsdelivr.net/npm/@fontsource/{fam}@5/files/{fam}-latin-{w}-{s}.woff'
FUENTES = [('cormorant-garamond', '700', 'normal'), ('cormorant-garamond', '600', 'italic'),
           ('dm-sans', '400', 'normal'), ('dm-sans', '600', 'normal'),
           ('barlow', '400', 'normal'), ('barlow', '500', 'normal'), ('barlow', '600', 'normal')]

def cargar_fuentes():
    try:
        from fontTools.ttLib import TTFont
    except ImportError:
        import subprocess, sys
        subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'fonttools'], check=False)
        from fontTools.ttLib import TTFont
    os.makedirs(FONT_DIR, exist_ok=True)
    for fam, w, s in FUENTES:
        ttf = os.path.join(FONT_DIR, f'{fam}-{w}-{s}.ttf')
        if not os.path.exists(ttf):
            try:
                woff = ttf.replace('.ttf', '.woff')
                urllib.request.urlretrieve(CDN.format(fam=fam, w=w, s=s), woff)
                f = TTFont(woff); f.flavor = None
                nombre = {'cormorant-garamond': 'Cormorant Garamond', 'dm-sans': 'DM Sans', 'barlow': 'Barlow'}[fam]
                for nid in (1, 16):   # misma familia para todos los pesos, así matplotlib los agrupa
                    f['name'].setName(nombre, nid, 3, 1, 0x409)
                f.save(ttf)
            except Exception as e:
                print(f'Fuente no disponible ({fam} {w}): se usa la de respaldo.')
                continue
        fm.fontManager.addfont(ttf)

cargar_fuentes()
SERIF = ['Cormorant Garamond', 'DejaVu Serif']   # titulares
SANS  = ['DM Sans', 'DejaVu Sans']               # texto
NUM   = ['Barlow', 'DejaVu Sans']                # cifras y etiquetas

plt.rcParams.update({
    'figure.facecolor': BG, 'axes.facecolor': BG, 'savefig.facecolor': BG,
    'text.color': FG, 'axes.labelcolor': FG, 'axes.edgecolor': P['SPINE'],
    'xtick.color': MUTED, 'ytick.color': MUTED, 'grid.color': GRID,
    'legend.facecolor': PANEL_BG, 'legend.edgecolor': GRID, 'legend.labelcolor': FG, 'legend.frameon': False,
    'font.family': SANS, 'font.size': 13, 'mathtext.fontset': 'dejavuserif',
    'axes.titlesize': 24, 'axes.titlelocation': 'left', 'axes.titlepad': 14,
    'axes.spines.top': False, 'axes.spines.right': False,
    'axes.grid': True, 'grid.alpha': 1.0, 'grid.linestyle': '-', 'grid.linewidth': 0.7,
})

EYEBROW = 'CÓMO NO QUEBRAR · VAR Y EXPECTED SHORTFALL'

def marca(ax, titulo, eyebrow=None):
    """Titular serif Imperial + eyebrow Barlow en oro encima (H3 Castle) + cifras de ejes en Barlow."""
    ax.set_title(titulo, fontfamily=SERIF, fontweight='bold', color=TITLE, pad=12)
    ax.annotate(eyebrow or EYEBROW, xy=(0, 1), xycoords='axes fraction', xytext=(0, 46), textcoords='offset points',
                fontfamily=NUM, fontweight='semibold', fontsize=11, color=GOLD_TXT, va='bottom', ha='left')
    ax.tick_params(labelfontfamily=NUM, labelsize=12)

def rotulo(ax, texto='Simulación'):
    ax.text(0.99, 0.02, texto, transform=ax.transAxes, ha='right', fontsize=10, color=MUTED)

print(f'Setup listo · opción de color: {OPCION}')

## Datos y funciones de cálculo

Todo lo que sigue usa **precios reales**, nada simulado:

- **Coin Metrics** (`PriceUSD`, cierre diario de BTC/USD, repositorio público en GitHub). Es la misma serie con la que se escribió el guion, así que las cifras coinciden con lo que se dice en el video.
- **Yahoo Finance** (vía `yfinance`) para cualquier otro activo y para BTC actualizado a hoy. Si Yahoo no responde y el activo es BTC, el notebook usa Coin Metrics.

Las pérdidas se expresan en positivo: un VaR de 9 % significa una caída de 9 %.

In [ ]:
import subprocess, sys
from scipy.stats import norm

COINMETRICS_URL = 'https://raw.githubusercontent.com/coinmetrics/data/master/csv/btc.csv'

def fpct(x, d=1, signo=False):
    """0.1252 -> '12,5 %' (coma decimal, signo menos tipográfico)."""
    s = f"{x * 100:{'+' if signo else ''}.{d}f}".replace('.', ',').replace('-', '−')
    return s + ' %'

def fnum(x, d=0):
    """12345.6 -> '12.346'"""
    return f'{x:,.{d}f}'.replace(',', '_').replace('.', ',').replace('_', '.')

def btc_coinmetrics(desde):
    df = pd.read_csv(COINMETRICS_URL, usecols=['time', 'PriceUSD'], parse_dates=['time'])
    s = df.dropna().set_index('time')['PriceUSD'].rename('Close').sort_index()
    return s[s.index >= pd.Timestamp(desde) - pd.Timedelta(days=10)]

def precios_yahoo(ticker, desde):
    try:
        import yfinance as yf
    except ImportError:
        subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'yfinance'], check=False)
        import yfinance as yf
    inicio = (pd.Timestamp(desde) - pd.Timedelta(days=10)).strftime('%Y-%m-%d')
    fin = (pd.Timestamp.today() + pd.Timedelta(days=1)).strftime('%Y-%m-%d')
    raw = yf.download(ticker, start=inicio, end=fin, auto_adjust=True, progress=False)
    if raw is None or len(raw) == 0:
        raise ValueError(f'Yahoo Finance no devolvió datos para {ticker}')
    if isinstance(raw.columns, pd.MultiIndex):
        raw.columns = raw.columns.get_level_values(0)
    s = raw['Close']
    if isinstance(s, pd.DataFrame):
        s = s.iloc[:, 0]
    s = s.dropna().rename('Close')
    s.index = pd.to_datetime(s.index)
    if s.index.tz is not None:
        s.index = s.index.tz_localize(None)
    return s.sort_index()

def precios(ticker, desde, fuente='yahoo'):
    """Precios de cierre diarios. fuente='coinmetrics' solo sirve para BTC."""
    if fuente == 'coinmetrics':
        return btc_coinmetrics(desde)
    try:
        return precios_yahoo(ticker, desde)
    except Exception as e:
        if ticker.upper().startswith('BTC'):
            print(f'Yahoo Finance no respondió ({e}). Uso Coin Metrics para BTC.')
            return btc_coinmetrics(desde)
        raise

def retornos(s, desde):
    r = s.pct_change().dropna()
    return r[r.index >= pd.Timestamp(desde)]

def medir_cola(r, confianza=0.99):
    """VaR y Expected Shortfall históricos (pérdidas en positivo) y la comparación con la campana de Gauss."""
    q = r.quantile(1 - confianza)
    cola = r[r <= q]
    mu, sd = r.mean(), r.std()
    z = norm.ppf(confianza)
    var_n = -(mu - z * sd)
    return dict(
        var=-q, es=-cola.mean(), dias_cola=len(cola),
        var_normal=var_n, es_normal=-(mu - sd * norm.pdf(z) / (1 - confianza)),
        excesos_normal=(r < -var_n).mean(),
        peor=-r.min(), fecha_peor=r.idxmin(), sigmas_peor=(mu - r.min()) / sd,
        n=len(r), anios=(r.index[-1] - r.index[0]).days / 365.25,
    )

def es_movil(r, ventana, confianza=0.99):
    """Expected Shortfall histórico en una ventana móvil de 'ventana' observaciones."""
    return r.rolling(ventana).apply(lambda x: -x[x <= np.quantile(x, 1 - confianza)].mean(), raw=True).dropna()

def fuente(fig, texto):
    """Fuente de los datos, abajo a la derecha y fuera del área del gráfico."""
    fig.text(0.99, 0.008, texto, ha='right', va='bottom', fontsize=10, color=MUTED)

print('Funciones de datos listas.')

# Parte 1 · Las gráficas del video

## Las cifras del guion

Esta celda descarga BTC/USD desde el 1 de enero de 2020 y calcula **todas las cifras que se dicen al aire**. Léelas de la salida, no del guion: si cambias `FUENTE_VIDEO` a `'yahoo'`, los datos llegan hasta hoy y los números se mueven un poco.

El "día malo" del video es una caída de 9 %: el número redondo que el trader tiene en la cabeza. Equivale más o menos al percentil 99, y al final de la salida está también la definición estándar al 99 %.

In [ ]:
# =====================  PARÁMETROS DEL VIDEO  =====================
FUENTE_VIDEO   = 'coinmetrics'  # 'coinmetrics' = los mismos datos del guion · 'yahoo' = BTC-USD actualizado a hoy
INICIO         = '2020-01-01'
UMBRAL         = 0.09           # el "día malo" que el trader cree que es lo peor
APALANCAMIENTO = 5
CUENTA_EJEMPLO = 10_000
RIESGO_EJEMPLO = 0.02           # lo que acepta perder en un día malo
# ==================================================================

px_btc = precios('BTC-USD', INICIO, FUENTE_VIDEO)
r_btc = retornos(px_btc, INICIO)
anios = (r_btc.index[-1] - r_btc.index[0]).days / 365.25
FUENTE_TXT = 'Coin Metrics' if FUENTE_VIDEO == 'coinmetrics' else 'Yahoo Finance'
DATOS_TXT = f"Datos reales · BTC/USD diario · {FUENTE_TXT} · {r_btc.index[0]:%m/%Y}–{r_btc.index[-1]:%m/%Y}"

cola = r_btc[r_btc < -UMBRAL]
V = dict(
    a_salvo    = (r_btc >= -UMBRAL).mean(),
    dias_cola  = len(cola),
    por_anio   = len(cola) / anios,
    es_umbral  = -cola.mean(),
    peor       = -r_btc.min(),
    fecha_peor = r_btc.idxmin(),
)
V['es_apal'] = APALANCAMIENTO * V['es_umbral']
V['posicion'] = CUENTA_EJEMPLO * RIESGO_EJEMPLO / V['es_umbral']
V['perdida_peor_regla'] = V['posicion'] * V['peor']
std = medir_cola(r_btc, 0.99)

print(f"BTC/USD · {r_btc.index[0]:%d-%m-%Y} a {r_btc.index[-1]:%d-%m-%Y} · {len(r_btc):,} días · fuente: {FUENTE_TXT}".replace(',', '.'))
print('\nCIFRAS PARA DECIR AL AIRE')
print(f"  Días con caída menor a {fpct(UMBRAL, 0)}:          {fpct(V['a_salvo'])}")
print(f"  Días que la superan:                 {V['dias_cola']} ({fnum(V['por_anio'], 1)} por año en promedio)")
print(f"  Caída promedio esos días (ES):       {fpct(V['es_umbral'])}")
print(f"  Con {APALANCAMIENTO}× de apalancamiento:            {fpct(V['es_apal'])} de la cuenta")
print(f"  Peor día:                            {fpct(-V['peor'])} el {V['fecha_peor']:%d-%m-%Y} (con {APALANCAMIENTO}×: {fpct(APALANCAMIENTO * V['peor'], 0)} → liquidado)")
print(f"  Regla: {fnum(CUENTA_EJEMPLO)} × {fpct(RIESGO_EJEMPLO, 0)} ÷ {fpct(V['es_umbral'])} = {fnum(V['posicion'])} USD")
print(f"  Esa posición en el peor día:         −{fnum(V['perdida_peor_regla'])} USD = {fpct(V['perdida_peor_regla'] / CUENTA_EJEMPLO)} de la cuenta")
print(f"\nDefinición estándar al 99 %: VaR {fpct(std['var'])} · ES {fpct(std['es'])} ({std['dias_cola']} días)")

## 1. Matemática (0:21–0:27) — Casi siempre, BTC cae menos de 9 %

**Datos reales.** Histograma de todos los retornos diarios de BTC desde 2020. La línea dorada es el "día malo" de 9 %: todo lo que está a su derecha es lo que el trader espera, y la cola naranja es lo que no.

Entra en: *"tienes razón: el noventa y nueve por ciento de los días cae menos de nueve"*. El porcentaje grande se lee de la figura.

In [ ]:
x = r_btc * 100
lim = 15
fig, ax = plt.subplots(figsize=(11, 6.4))
bins = np.arange(-lim, lim + 0.5, 0.5)
n, b, barras = ax.hist(x[x.between(-lim, lim)], bins=bins, color=MAIN, alpha=0.75, edgecolor=BG, linewidth=0.6)
for barra, izq in zip(barras, b[:-1]):
    if izq < -UMBRAL * 100:
        barra.set_facecolor(ALERT); barra.set_alpha(0.9)
ax.axvline(-UMBRAL * 100, color=GOLD, lw=2.5)
ax.set_xlim(-lim, lim)
ytop = ax.get_ylim()[1]
ax.text(-UMBRAL * 100 + 0.3, ytop * 0.93, f"−{UMBRAL * 100:.0f} %: el día malo", fontfamily=NUM, fontweight='semibold',
        fontsize=14, color=GOLD_TXT, ha='left', va='top')

ax.text(0.73, 0.70, fpct(V['a_salvo']), transform=ax.transAxes, fontfamily=NUM, fontweight='semibold', fontsize=46,
        color=MAIN, ha='center')
ax.text(0.73, 0.62, f"de los días cae menos de {UMBRAL * 100:.0f} %", transform=ax.transAxes, fontsize=14, color=FG, ha='center')
ax.text(0.03, 0.42, f"{fpct(1 - V['a_salvo'])}\n{V['dias_cola']} días", transform=ax.transAxes, fontfamily=NUM,
        fontweight='semibold', fontsize=18, color=ALERT, ha='left')
fuera = x[x < -lim]
if len(fuera):
    ax.text(0.01, 0.10, f"+ {len(fuera)} días más allá de −{lim} %\n(el peor: {fpct(-V['peor'])})", transform=ax.transAxes,
            fontsize=11, color=MUTED, ha='left')
ax.set_xlabel('Retorno diario (%)')
ax.set_ylabel('Número de días')
marca(ax, f'Casi siempre, BTC cae menos de {UMBRAL * 100:.0f} %')
fuente(fig, DATOS_TXT)
plt.tight_layout(rect=(0, 0.035, 1, 1)); plt.show()
print(f"Días con caída menor a {fpct(UMBRAL, 0)}: {fpct(V['a_salvo'])} · días en la cola: {V['dias_cola']}")

## 2. Matemática y nombre (0:28–0:41) — El VaR dice dónde empieza. El ES, cuánto mide.

**Datos reales.** Cada barra es uno de los días en que BTC cayó más de 9 %, ordenados de menor a mayor. La línea azul punteada es el **Value at Risk** (dónde empieza el desastre). La dorada es el **Expected Shortfall**: el promedio de esas caídas (cuánto mide).

Entra en: *"Pero cuando lo pasa, cae en promedio más de doce: con tu apalancamiento, más del sesenta por ciento de la cuenta"* y se queda en pantalla con *"Ese nueve es el Value at Risk…"*.

In [ ]:
cola_ord = cola.sort_values(ascending=False)          # de la menor caída a la mayor
caidas = -cola_ord.values * 100
xs = np.arange(len(caidas))
fig, ax = plt.subplots(figsize=(11, 6.4))
ax.bar(xs, caidas, color=ALERT, alpha=0.85, width=0.72, edgecolor=BG)
ax.axhline(UMBRAL * 100, color=MAIN, lw=2, ls='--')
ax.axhline(V['es_umbral'] * 100, color=GOLD, lw=2.8)
trans = ax.get_yaxis_transform()
ax.text(1.01, UMBRAL * 100 - 0.4, f"VaR\n{fpct(UMBRAL, 1)}", transform=trans, va='top', ha='left', fontfamily=NUM,
        fontweight='semibold', fontsize=13, color=MAIN, clip_on=False)
ax.text(1.01, V['es_umbral'] * 100 + 0.4, f"ES\n{fpct(V['es_umbral'])}", transform=trans, va='bottom', ha='left',
        fontfamily=NUM, fontweight='semibold', fontsize=13, color=GOLD_TXT, clip_on=False)

ax.text(0.03, 0.93, 'VaR = dónde empieza el desastre', transform=ax.transAxes, fontsize=14, color=MAIN, va='top')
ax.text(0.03, 0.85, 'Expected Shortfall = cuánto mide (promedio de la cola)', transform=ax.transAxes, fontsize=14,
        color=GOLD_TXT, va='top')
ax.text(0.03, 0.70, f"× {APALANCAMIENTO} de apalancamiento = {fpct(V['es_apal'])} de la cuenta", transform=ax.transAxes,
        fontfamily=NUM, fontweight='semibold', fontsize=17, color=ALERT, va='top',
        bbox=dict(boxstyle='round,pad=0.5', facecolor=PANEL_BG, edgecolor=ALERT, lw=1.2))
ax.set_xticks(xs)
ax.set_xticklabels([d.strftime('%d-%m-%y') for d in cola_ord.index], rotation=60, ha='right', fontsize=9)
ax.set_xlim(-0.6, len(caidas) - 0.4)
ax.set_ylim(0, max(caidas) * 1.08)
ax.set_ylabel('Caída del día (%)')
ax.grid(axis='x', visible=False)
marca(ax, 'El VaR dice dónde empieza. El ES, cuánto mide.')
fuente(fig, DATOS_TXT)
plt.tight_layout(rect=(0, 0.035, 1, 1)); plt.show()
print(f"{V['dias_cola']} días con caída mayor a {fpct(UMBRAL, 0)} · promedio (ES): {fpct(V['es_umbral'])} · con {APALANCAMIENTO}×: {fpct(V['es_apal'])}")

## 3. Matemática (0:32–0:36) — Un solo día se lleva la cuenta

**Datos reales.** Cada línea vertical es un día de BTC desde 2020. En naranja, los días que cayeron más de 9 %. La línea punteada marca dónde una posición con 5× queda liquidada: una caída de alrededor de 20 % borra el margen (en la práctica, el exchange liquida un poco antes por el margen de mantenimiento).

Entra en: *"Y en el crash del covid, treinta y siete. Liquidado."* Deja la figura quieta durante la pausa de 1 s.

In [ ]:
y = r_btc * 100
normal = y >= -UMBRAL * 100
fig, ax = plt.subplots(figsize=(11, 6.4))
ax.vlines(y.index[normal], 0, y[normal], color=MAIN, alpha=0.40, lw=0.8)
ax.vlines(y.index[~normal], 0, y[~normal], color=ALERT, lw=2.0)
ax.axhline(-UMBRAL * 100, color=GOLD, lw=2.2)
liq = -100 / APALANCAMIENTO
ax.axhline(liq, color=ALERT, lw=1.6, ls='--')
ax.text(y.index[-1], -UMBRAL * 100 + 0.8, f"−{UMBRAL * 100:.0f} %: el día malo que esperabas", ha='right', va='bottom',
        fontsize=12, color=GOLD_TXT)
ax.text(y.index[-1], liq - 0.8, f"≈ −{100 / APALANCAMIENTO:.0f} %: con {APALANCAMIENTO}× te liquidan", ha='right', va='top',
        fontsize=12, color=ALERT)
d = V['fecha_peor']
ax.annotate(f"{d:%d-%m-%Y}  {fpct(-V['peor'])}", xy=(d, -V['peor'] * 100), xytext=(28, 4), textcoords='offset points',
            fontfamily=NUM, fontweight='semibold', fontsize=16, color=ALERT, va='center',
            arrowprops=dict(arrowstyle='-', color=ALERT, lw=1.2))
ax.annotate('LIQUIDADO', xy=(d, -V['peor'] * 100), xytext=(28, -22), textcoords='offset points',
            fontfamily=NUM, fontweight='semibold', fontsize=20, color=ALERT, va='center')
ax.set_ylim(min(-V['peor'] * 100 * 1.22, liq - 6), max(y.max() * 1.1, 10))
ax.set_ylabel('Retorno diario (%)')
ax.grid(axis='x', visible=False)
marca(ax, 'Un solo día se lleva la cuenta')
fuente(fig, DATOS_TXT)
plt.tight_layout(rect=(0, 0.035, 1, 1)); plt.show()
print(f"Peor día: {fpct(-V['peor'])} el {d:%d-%m-%Y} · con {APALANCAMIENTO}× la pérdida sería {fpct(APALANCAMIENTO * V['peor'], 0)} → liquidación")

## 4. Veredicto (0:41–0:46) — No es mala suerte

**Datos reales.** Cuántos días por año cayó BTC más de 9 %. La línea dorada es el promedio de todo el periodo. Ojo: no llegan repartidos. Se amontonan en los años de alta volatilidad, y en los años tranquilos puede no aparecer ninguno; por eso el que se confía en un año calmo es el que más se expone cuando vuelve la racha.

Entra en: *"no es mala suerte: eso pasa unas tres veces al año"*. Revisa la salida antes de grabar (ver la nota de la entrega).

In [ ]:
anios_idx = range(r_btc.index[0].year, r_btc.index[-1].year + 1)
conteo = cola.groupby(cola.index.year).size().reindex(anios_idx, fill_value=0)
etiquetas = [str(a) for a in conteo.index]
parcial = r_btc.index[-1] < pd.Timestamp(f'{r_btc.index[-1].year}-12-31')
if parcial:
    etiquetas[-1] += '*'
fig, ax = plt.subplots(figsize=(11, 6.2))
barras = ax.bar(etiquetas, conteo.values, color=ALERT, alpha=0.85, width=0.6, edgecolor=BG)
for barra, v in zip(barras, conteo.values):
    ax.text(barra.get_x() + barra.get_width() / 2, v + 0.15, str(v), ha='center', va='bottom', fontfamily=NUM,
            fontweight='semibold', fontsize=16, color=FG)
ax.axhline(V['por_anio'], color=GOLD, lw=2.5)
ax.text(len(etiquetas) - 0.5, V['por_anio'] + 0.15, f"promedio: {fnum(V['por_anio'], 1)} por año", ha='right', va='bottom',
        fontfamily=NUM, fontweight='semibold', fontsize=14, color=GOLD_TXT)
ax.set_ylim(0, max(conteo.max(), V['por_anio']) * 1.25 + 0.5)
ax.set_ylabel(f'Días con caída mayor a {UMBRAL * 100:.0f} %')
ax.grid(axis='x', visible=False)
marca(ax, f"{V['dias_cola']} días en {fnum(anios, 1)} años, y llegan en racha")
fuente(fig, DATOS_TXT + (f" · * {r_btc.index[-1].year} hasta el {r_btc.index[-1]:%d-%m-%Y}" if parcial else ''))
plt.tight_layout(rect=(0, 0.035, 1, 1)); plt.show()
print('Días con caída mayor a', fpct(UMBRAL, 0), 'por año:')
print(conteo.to_string())
print(f"Promedio: {fnum(V['por_anio'], 1)} por año")

## 5. Control (0:46–0:56) — La regla del Expected Shortfall

**Datos reales, cuenta ilustrativa.** Dos traders con una cuenta de 10.000 USD frente a las mismas caídas reales de BTC:

- **El de la escena:** compra BTC por 5 veces su cuenta.
- **El de la regla:** posición máxima = lo que acepta perder en un día malo (2 %, o sea 200 USD) ÷ Expected Shortfall.

Barras = porcentaje de la cuenta que pierde cada uno en un día de cola promedio y en el peor día. Sin comisiones ni funding.

Entra en: *"Si aceptas perder dos por ciento, el crash del covid te cuesta seis. No la cuenta."* La regla y sus números también son el frame guardable.

In [ ]:
exp_trader = CUENTA_EJEMPLO * APALANCAMIENTO
exp_regla = V['posicion']
escenarios = ['Un día de cola\n(promedio)', f"El peor día\n({V['fecha_peor']:%d-%m-%Y})"]
caidas_esc = [V['es_umbral'], V['peor']]
p_trader = [min(exp_trader * c, CUENTA_EJEMPLO) / CUENTA_EJEMPLO for c in caidas_esc]
p_regla = [exp_regla * c / CUENTA_EJEMPLO for c in caidas_esc]

fig, ax = plt.subplots(figsize=(11, 6.4))
yy = np.arange(len(escenarios))
h = 0.36
ax.barh(yy - h / 2, np.array(p_trader) * 100, height=h, color=ALERT, alpha=0.9,
        label=f'{APALANCAMIENTO}× la cuenta ({fnum(exp_trader)} USD en BTC)')
ax.barh(yy + h / 2, np.array(p_regla) * 100, height=h, color=MAIN, alpha=0.9,
        label=f'Regla del ES ({fnum(exp_regla)} USD en BTC)')
for i in range(len(escenarios)):
    txt = 'LIQUIDADO' if p_trader[i] >= 1 else fpct(p_trader[i])
    ax.text(p_trader[i] * 100 + 1.5, yy[i] - h / 2, txt, va='center', fontfamily=NUM, fontweight='semibold',
            fontsize=16, color=ALERT)
    ax.text(p_regla[i] * 100 + 1.5, yy[i] + h / 2, fpct(p_regla[i]), va='center', fontfamily=NUM, fontweight='semibold',
            fontsize=16, color=GOLD_TXT if i == len(escenarios) - 1 else MAIN)
ax.set_yticks(yy); ax.set_yticklabels(escenarios, fontsize=13)
ax.tick_params(axis='y', labelfontfamily=SANS)
ax.set_ylim(len(escenarios) - 1 + 1.05, -0.6)   # eje invertido, con una franja libre abajo para la regla
ax.set_xlim(0, 125)
ax.set_xlabel('Porcentaje de la cuenta perdido en el día')
ax.grid(axis='y', visible=False)
ax.legend(loc='upper right', fontsize=12, frameon=True)
regla = (f"Posición máx = pérdida que aceptas ÷ ES\n"
         f"{fnum(CUENTA_EJEMPLO)} × {fpct(RIESGO_EJEMPLO, 0)} ÷ {fpct(V['es_umbral'])} = {fnum(exp_regla)} USD")
ax.text(0.98, 0.03, regla, transform=ax.transAxes, ha='right', va='bottom', fontsize=13, color=FG, linespacing=1.6,
        bbox=dict(boxstyle='round,pad=0.6', facecolor=PANEL_BG, edgecolor=GRID))
marca(ax, f"Mismo crash: {fpct(p_regla[-1], 0)} de la cuenta, o la cuenta entera")
fuente(fig, 'Caídas reales de BTC · cuenta de 10.000 USD ilustrativa · sin comisiones ni funding')
plt.tight_layout(rect=(0, 0.035, 1, 1)); plt.show()

print('FRAME GUARDABLE')
print('  POSICIÓN MÁX = PÉRDIDA QUE ACEPTAS ÷ ES')
print(f"  BTC {r_btc.index[0]:%Y}–{r_btc.index[-1]:%Y}:  VaR ≈ {fpct(UMBRAL, 0)}  ·  ES ≈ {fpct(V['es_umbral'])}")
print(f"  Cuenta {fnum(CUENTA_EJEMPLO)} · aceptas {fnum(CUENTA_EJEMPLO * RIESGO_EJEMPLO)} ({fpct(RIESGO_EJEMPLO, 0)}) → {fnum(exp_regla)} USD")
print(f"  Peor día ({fpct(-V['peor'])}): −{fnum(V['perdida_peor_regla'])} USD = {fpct(-p_regla[-1])}")

# Parte 2 · Tu calculadora de tamaño de posición

La regla del video, aplicada a tu cuenta y al activo que operes:

> **Posición máxima = lo que aceptas perder en un día malo ÷ Expected Shortfall**

Cambia los valores de la celda de abajo y ejecútala:

- `ACTIVO`: cualquier ticker de Yahoo Finance (`'ETH-USD'`, `'SOL-USD'`, `'QQQ'`, `'SPY'`, `'AAPL'`, `'EURUSD=X'`, `'GC=F'`…).
- `DESDE`: desde cuándo medir. Si la ventana incluye una crisis (2020, 2022), la medida es más conservadora, que es lo que quieres para dimensionar.
- `CONFIANZA`: `0.99` mide el 1 % de días peores; `0.975` es el nivel que usa Basilea para los bancos.
- `CUENTA` y `RIESGO_DIA`: tu cuenta y la fracción que aceptas perder en un día malo.
- `APALANCAMIENTO_HOY`: el que usas hoy, para comparar.

En el video se usó el umbral redondo de 9 % para BTC. Aquí se usa el percentil exacto de la `CONFIANZA` que elijas, así que el ES puede salir un poco distinto.

In [ ]:
# ========================  TUS DATOS  ========================
ACTIVO             = 'BTC-USD'
DESDE              = '2020-01-01'
CONFIANZA          = 0.99
CUENTA             = 10_000
RIESGO_DIA         = 0.02    # 0.02 = aceptas perder hasta 2 % de la cuenta en un día malo
APALANCAMIENTO_HOY = 5       # 1 = sin apalancamiento
# =============================================================

px_tu = precios(ACTIVO, DESDE)
r_tu = retornos(px_tu, DESDE)
m = medir_cola(r_tu, CONFIANZA)
nivel = fpct(CONFIANZA, 1 if CONFIANZA * 1000 % 10 else 0)
cola_txt = fpct(1 - CONFIANZA, 1 if (1 - CONFIANZA) * 1000 % 10 > 0.5 else 0)

posicion_max = CUENTA * RIESGO_DIA / m['es']
veces = posicion_max / CUENTA
exp_hoy = CUENTA * APALANCAMIENTO_HOY
obs_anio = len(r_tu) / m['anios']                 # ~365 en cripto, ~252 en acciones
ventana = int(round(obs_anio))
es_ult = medir_cola(r_tu.iloc[-ventana:], CONFIANZA)['es'] if len(r_tu) > 2 * ventana else m['es']

linea = '─' * 64
print(f"{ACTIVO} · {r_tu.index[0]:%d-%m-%Y} a {r_tu.index[-1]:%d-%m-%Y} · {fnum(m['n'])} días")
print(linea)
print(f"{'VaR al ' + nivel + ':':<25}{fpct(m['var'])}   (en el {cola_txt} de días peores, la caída pasa de aquí)")
print(f"{'Expected Shortfall:':<25}{fpct(m['es'])}   (la caída promedio de esos días)")
print(f"{'Peor día:':<25}{fpct(-m['peor'])}   ({m['fecha_peor']:%d-%m-%Y})")
print(linea)
print(f"{'TU POSICIÓN MÁXIMA:':<25}{fnum(CUENTA)} × {fpct(RIESGO_DIA, 1)} ÷ {fpct(m['es'])} = {fnum(posicion_max)}")
if veces >= 1:
    print(f"{'':<25}{fnum(veces, 2)}× tu cuenta: apalancamiento máximo de {fnum(veces, 1)}×")
else:
    print(f"{'':<25}{fpct(veces)} de tu cuenta, sin apalancamiento")
print(f"{'Con esa posición:':<25}un día de cola te cuesta {fpct(RIESGO_DIA, 1)}; el peor día, {fpct(posicion_max * m['peor'] / CUENTA)}")
print(linea)
p_es_hoy = exp_hoy * m['es'] / CUENTA
p_peor_hoy = exp_hoy * m['peor'] / CUENTA
print(f"CON TU {APALANCAMIENTO_HOY}× DE HOY ({fnum(exp_hoy)} en {ACTIVO}):")
print(f"  un día de cola:        {'LIQUIDADO' if p_es_hoy >= 1 else fpct(p_es_hoy) + ' de la cuenta'}")
print(f"  el peor día:           {'LIQUIDADO' if p_peor_hoy >= 1 else fpct(p_peor_hoy) + ' de la cuenta'}")
print(f"  estás {fnum(exp_hoy / posicion_max, 1)} veces por encima de tu posición máxima" if exp_hoy > posicion_max
      else '  estás dentro de tu posición máxima')
print(linea)
print(f"RECALCULA: el ES del último año es {fpct(es_ult)} (toda la ventana: {fpct(m['es'])}).")
if es_ult > m['es']:
    print(f"  El riesgo reciente es mayor: con ese ES tu posición máxima baja a {fnum(CUENTA * RIESGO_DIA / es_ult)}.")
print(f"CAMPANA DE GAUSS: daría un VaR de {fpct(m['var_normal'])}, superado el {fpct(m['excesos_normal'])} de los días "
      f"(debería ser {cola_txt}). El peor día fue de {fnum(m['sigmas_peor'], 1)} desviaciones estándar.")
print('\nContenido educativo, no es recomendación de inversión.')

## 6. Tu activo: dónde empieza el desastre y cuánto mide

**Datos reales** del activo que pusiste arriba. La línea azul punteada es tu VaR y la dorada tu Expected Shortfall. La tarjeta resume tu posición máxima.

In [ ]:
x = r_tu * 100
lim = max(abs(np.quantile(x, 0.001)), abs(np.quantile(x, 0.999))) * 1.15
fig, ax = plt.subplots(figsize=(11, 6.4))
bins = np.linspace(-lim, lim, 70)
n, b, barras = ax.hist(x[x.between(-lim, lim)], bins=bins, color=MAIN, alpha=0.75, edgecolor=BG, linewidth=0.5)
for barra, der in zip(barras, b[1:]):
    if der <= -m['var'] * 100:
        barra.set_facecolor(ALERT); barra.set_alpha(0.9)
ax.axvline(-m['var'] * 100, color=MAIN, lw=2, ls='--')
ax.axvline(-m['es'] * 100, color=GOLD, lw=2.6)
ax.set_xlim(-lim, lim)
ytop = ax.get_ylim()[1]
ax.text(-m['var'] * 100 + lim * 0.015, ytop * 0.97, f"VaR {fpct(m['var'])}", ha='left', va='top', fontfamily=NUM,
        fontweight='semibold', fontsize=13, color=MAIN)
ax.text(-m['es'] * 100 - lim * 0.015, ytop * 0.97, f"ES {fpct(m['es'])}", ha='right', va='top', fontfamily=NUM,
        fontweight='semibold', fontsize=13, color=GOLD_TXT)
tarjeta = (f"Posición máxima\n{fnum(posicion_max)}\n"
           f"{fnum(veces, 2)}× tu cuenta · riesgo {fpct(RIESGO_DIA, 1)}")
ax.text(0.97, 0.93, tarjeta, transform=ax.transAxes, ha='right', va='top', fontfamily=NUM, fontweight='semibold',
        fontsize=16, color=TITLE, linespacing=1.5,
        bbox=dict(boxstyle='round,pad=0.7', facecolor=PANEL_BG, edgecolor=GRID))
fuera = (x < -lim).sum()
if fuera:
    ax.text(0.01, 0.10, f"+ {fuera} días fuera de escala\n(el peor: {fpct(-m['peor'])})", transform=ax.transAxes,
            fontsize=11, color=MUTED)
ax.set_xlabel('Retorno diario (%)')
ax.set_ylabel('Número de días')
marca(ax, f"{ACTIVO}: VaR {fpct(m['var'])} · ES {fpct(m['es'])}", eyebrow=f'TU CALCULADORA · CONFIANZA {nivel}')
fuente(fig, f"Datos reales · {ACTIVO} diario · {r_tu.index[0]:%m/%Y}–{r_tu.index[-1]:%m/%Y}")
plt.tight_layout(rect=(0, 0.035, 1, 1)); plt.show()

## 7. Por qué hay que recalcularlo

**Datos reales.** El Expected Shortfall medido en ventanas móviles de un año. No es un número fijo: sube cuando el mercado se pone violento, y ahí es cuando tu posición máxima tiene que bajar.

In [ ]:
esm = es_movil(r_tu, ventana, CONFIANZA) * 100
fig, ax = plt.subplots(figsize=(11, 6.2))
ax.fill_between(esm.index, 0, esm.values, color=MAIN, alpha=0.14)
ax.plot(esm.index, esm.values, color=MAIN, lw=2.4)
ax.axhline(m['es'] * 100, color=GOLD, lw=2.2)
ax.text(esm.index[0], m['es'] * 100, f"  toda la ventana: {fpct(m['es'])}", va='bottom', ha='left', fontfamily=NUM,
        fontweight='semibold', fontsize=12, color=GOLD_TXT)
imax, imin = esm.idxmax(), esm.idxmin()
ax.annotate(f"máx. {fnum(esm.max(), 1)} %", xy=(imax, esm.max()), xytext=(10, 6), textcoords='offset points',
            fontfamily=NUM, fontweight='semibold', fontsize=13, color=ALERT)
ax.annotate(f"mín. {fnum(esm.min(), 1)} %", xy=(imin, esm.min()), xytext=(10, -16), textcoords='offset points',
            fontfamily=NUM, fontweight='semibold', fontsize=13, color=MAIN)
ax.annotate(f"hoy: {fnum(esm.iloc[-1], 1)} %", xy=(esm.index[-1], esm.iloc[-1]), xytext=(-8, 14), textcoords='offset points',
            ha='right', fontfamily=NUM, fontweight='semibold', fontsize=13, color=FG)
ax.set_ylim(0, esm.max() * 1.2)
ax.set_ylabel(f'Expected Shortfall al {nivel}, ventana de {ventana} días (%)')
marca(ax, 'El ES cambia con el mercado: recalcúlalo', eyebrow='TU CALCULADORA · ES EN VENTANA MÓVIL')
fuente(fig, f"Datos reales · {ACTIVO} · ventana móvil de {ventana} observaciones")
plt.tight_layout(rect=(0, 0.035, 1, 1)); plt.show()
print(f"ES en ventana de {ventana} días · mínimo {fnum(esm.min(), 1)} % ({imin:%m-%Y}) · máximo {fnum(esm.max(), 1)} % ({imax:%m-%Y}) · último {fnum(esm.iloc[-1], 1)} %")

## Extra · Compara varios activos

La misma regla para una lista de activos, con tu cuenta y tu riesgo de arriba. Sirve para ver de un vistazo cuánto puedes tener en cada uno.

In [ ]:
ACTIVOS = ['BTC-USD', 'ETH-USD', 'SOL-USD', 'QQQ', 'SPY', 'GC=F']

filas = []
for t in ACTIVOS:
    try:
        rr = retornos(precios(t, DESDE), DESDE)
        mm = medir_cola(rr, CONFIANZA)
        pos = CUENTA * RIESGO_DIA / mm['es']
        filas.append({'Activo': t, f'VaR {nivel}': fpct(mm['var']), 'Expected Shortfall': fpct(mm['es']),
                      'Peor día': fpct(-mm['peor']), 'Posición máxima': fnum(pos), 'Veces la cuenta': fnum(pos / CUENTA, 2)})
    except Exception as e:
        print(f'{t}: sin datos ({e})')
tabla = pd.DataFrame(filas).set_index('Activo')
print(f"Cuenta {fnum(CUENTA)} · riesgo por día malo {fpct(RIESGO_DIA, 1)} · desde {DESDE} · confianza {nivel}")
try:
    display(tabla)
except NameError:
    print(tabla.to_string())

## Guion de referencia

> **Matemática (0:21–0:36)**
> "Mira que, con datos reales desde 2020, tienes razón: el noventa y nueve por ciento de los días cae menos de nueve. **[figura 1]** Pero cuando lo pasa, cae en promedio más de doce: con tu apalancamiento, más del sesenta por ciento de la cuenta. **[figura 2]** Y en el crash del covid, treinta y siete. Liquidado." **[figura 3]**

> **Nombre (0:36–0:41)**
> "Ese nueve es el Value at Risk: te dice dónde empieza el desastre. El Expected Shortfall te dice cuánto mide." **[figura 2 sigue en pantalla]**

> **Veredicto (0:41–0:46)**
> "Déjame decirte, mi estimado, que no es mala suerte: eso pasa unas tres veces al año." **[figura 4]**

> **Control (0:46–0:56)**
> "¿Cómo lo controlamos? Tu posición máxima: lo que aceptas perder en un día malo, dividido entre el Expected Shortfall. Si aceptas perder dos por ciento, el crash del covid te cuesta seis. No la cuenta." **[figura 5 · frame guardable]**

El hook, la retención, la escena y el CTA se graban a cámara y no tienen material en este notebook.

*Contenido educativo, no es recomendación de inversión. Julián Castillo, trader cuantitativo de Castle Capital.*